# 02 · 웨이크 워드 → 제스처 → 모델 교체

**목적** — 01번 노트북 두 개와 `gesture_live.py` 를 한 파이프라인으로 묶는다.

```
        [항상]  마이크 ──── openWakeWord (CPU) ─────┐
        [항상]  cam0 ─┐                            │  event_q
        [항상]  cam1 ─┴─ 캡처 스레드 (최신 프레임)   ▼
                                            ┌─────────────┐
                                            │  DPU 워커   │  ← DPU 는 하나뿐이다
                                            └─────────────┘
   IDLE ──wake──▶ GESTURE (cam0, 제스처 xmodel)
                     │ open
                     ▼
                   TASK (cam1, 두 번째 xmodel) ──wake──▶ GESTURE
```

**핵심 제약은 DPU 가 하나라는 것이다.** 두 xmodel 을 동시에 올릴 수 없으므로
모드 전환이 곧 `overlay.load_model()` 호출이다. 비트스트림은 시작할 때 한 번만 올린다.

웨이크 워드는 CPU(ONNX)에서 돌기 때문에 DPU 교체 중에도 계속 듣는다.
이게 4.2(언제든 말하면 제스처 모드로 복귀)가 성립하는 이유다.

**두 번째 xmodel 은 지금 제스처 xmodel 을 그대로 다시 올린 대역(stand-in)이다.**
교체 동작·지연·복귀는 전부 진짜다. 진짜 2번 모델이 생기면 `TASK_XMODEL` 한 줄만 바꾼다.

전제: `00_dual_camera_check` PASS, `01_wakeword` PASS, `gesture_live.py` 단독 실행 확인.

## 1. 설정

In [ ]:
import os
import re
import time
import queue
import threading

import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# 제스처 수학(디코딩·시퀀스·게이트)은 전부 gesture_live.py 에 있다. 여기서 다시 쓰지 않는다.
from gesture_live import (
    Runner, GestureClassifier, SequenceBuffer,
    build_sequence, is_active, decode, draw,
    GESTURE_CLASSES,
)

# ---- 카메라 ----
CAM_A_DEV = "/dev/cam0"      # 사용자 — 제스처를 본다
CAM_B_DEV = "/dev/cam1"      # 대상   — 두 번째 모델이 본다
WIDTH, HEIGHT = 640, 480     # 00번 검증 스펙이자 제스처 학습 때의 프레임 비율
FOURCC = "MJPG"              # 고정. YUYV 는 이 카메라에서 30fps 가 안 나온다
TARGET_FPS = 30

DISPLAY_SCALE = 0.6
DISPLAY_FPS = 10
JPEG_QUALITY = 70

# ---- 마이크 (01_wakeword 3장 값 그대로) ----
MIC_INDEX = 5
SAMPLE_RATE, CHANNELS, DTYPE, CHUNK = 16000, 1, "int16", 1280
WAKEWORD_MODEL = "hey_jarvis"     # 커스텀 hey_kria.onnx 가 나오면 경로로 교체
WAKE_THRESHOLD = 0.5
WAKE_REFRACTORY = 2.0

# ---- DPU ----
BIT_PATH = "dpu.bit"
GESTURE_XMODEL = "./gesture_stage1_kv260.xmodel"
TASK_XMODEL = GESTURE_XMODEL      # ← 데모용 대역. 진짜 2번 모델이 생기면 이 줄만 바꾼다
GRU_PATH = "./gesture_stage2_gru.tflite"

# ---- 제스처 (gesture_live.py 기본값) ----
CONF = 0.15                  # 학습 시퀀스 추출값. 바꾸면 특징 분포가 어긋난다
WINDOW_S = 1.0
GRU_EVERY = 5
GESTURE_THRES = 0.70
GESTURE_REFRACTORY = 1.5
TRIGGER = "open"             # 2단계 제스처 이름. 1단계 손모양 open_palm 과 다른 것이다

assert TRIGGER in GESTURE_CLASSES, f"{TRIGGER} 는 2단계 클래스가 아니다: {GESTURE_CLASSES}"


def dev_index(path):
    real = os.path.realpath(path)
    m = re.search(r"(\d+)$", real)
    if not m:
        raise ValueError(f"video 인덱스를 찾을 수 없다: {path} -> {real}")
    return int(m.group(1))


try:
    CAM_A_ID, CAM_B_ID = dev_index(CAM_A_DEV), dev_index(CAM_B_DEV)
except (ValueError, OSError) as e:
    print(f"[WARN] 심볼릭 링크 해석 실패 ({e}). 인덱스를 직접 지정한다.")
    CAM_A_ID, CAM_B_ID = 0, 2

print(f"CAM_A = {CAM_A_DEV} -> /dev/video{CAM_A_ID}  (제스처)")
print(f"CAM_B = {CAM_B_DEV} -> /dev/video{CAM_B_ID}  (두 번째 모델)")
print(f"캡처 {WIDTH}x{HEIGHT} {FOURCC} @{TARGET_FPS}")
print(f"제스처 xmodel : {GESTURE_XMODEL}")
print(f"작업   xmodel : {TASK_XMODEL}"
      + ("   ← 같은 파일이다 (데모용 대역)" if TASK_XMODEL == GESTURE_XMODEL else ""))
print(f"웨이크 워드   : {WAKEWORD_MODEL}  threshold={WAKE_THRESHOLD}")

## 2. 캡처 스레드

`01_dual_camera_view.ipynb` 3장과 같다. `.ipynb` 는 import 가 안 되므로 옮겨 왔다.

카메라마다 스레드를 하나씩 둔다. 두 소비자(화면·DPU 워커)가 각자 `snapshot()` 으로
최신 프레임 한 장만 가져가므로 큐가 없고 지연이 쌓이지 않는다.

In [ ]:
def set_v4l2_ctrl(dev_id, name, value):
    import subprocess
    return subprocess.run(
        ["v4l2-ctl", "-d", f"/dev/video{dev_id}", "-c", f"{name}={value}"],
        capture_output=True, text=True,
    ).returncode == 0


class CamStream(threading.Thread):
    """카메라 한 대에서 계속 읽으며 최신 프레임과 실효 fps를 유지한다."""

    def __init__(self, dev_id, label):
        super().__init__(daemon=True)
        self.dev_id = dev_id
        self.label = label
        self.frame = None
        self.fps = 0.0
        self.reads = 0
        self.fails = 0
        self._lock = threading.Lock()
        self._running = False
        self.cap = None

    def open(self):
        cap = cv2.VideoCapture(self.dev_id, cv2.CAP_V4L2)
        if not cap.isOpened():
            cap.release()
            return False

        # FOURCC를 해상도보다 먼저 설정해야 한다
        cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*FOURCC))
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)
        cap.set(cv2.CAP_PROP_FPS, TARGET_FPS)
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)

        # 저조도에서 카메라가 프레임 주기를 2배로 늘리는 것을 막는다
        set_v4l2_ctrl(self.dev_id, "exposure_dynamic_framerate", 0)

        self.cap = cap
        return True

    def run(self):
        self._running = True
        n, t0 = 0, time.monotonic()
        while self._running:
            ok, f = self.cap.read()
            if ok and f is not None:
                with self._lock:
                    self.frame = f
                self.reads += 1
                n += 1
                if n >= 15:
                    now = time.monotonic()
                    self.fps = n / (now - t0)
                    n, t0 = 0, now
            else:
                self.fails += 1
                time.sleep(0.005)

    def snapshot(self):
        with self._lock:
            return None if self.frame is None else self.frame.copy()

    def stop(self):
        self._running = False
        self.join(timeout=2.0)
        if self.cap is not None:
            self.cap.release()
            self.cap = None

## 3. 상태 기계

파이프라인 전체의 전환 규칙이 이 네 줄이다. 하드웨어 없이 검증된다.

- `wake` 는 어느 상태에서든 제스처 모드로 보낸다 (요구사항 4.2)
- `open` 은 제스처 모드에서만 의미가 있다 — 제스처 모델이 안 돌고 있으면 나올 수 없는 이벤트다
- 두 이벤트 모두 같은 큐로 들어가 워커 한 곳에서만 소비된다. 그래서 경합이 없다

In [ ]:
def next_mode(mode, event):
    if event == "wake":
        return "GESTURE"
    if event == "open" and mode == "GESTURE":
        return "TASK"
    return mode


# --- 자체 점검 (하드웨어 불필요) ---
assert next_mode("IDLE", "wake") == "GESTURE"          # 1) 웨이크로 시작
assert next_mode("GESTURE", "open") == "TASK"          # 2) open 으로 모델 교체
assert next_mode("TASK", "wake") == "GESTURE"          # 3) 웨이크로 복귀
assert next_mode("GESTURE", "wake") == "GESTURE"       # 4) 이미 제스처면 무동작
assert next_mode("IDLE", "open") == "IDLE"             # 5) 대기 중 open 은 무시
assert next_mode("TASK", "open") == "TASK"             # 6) 작업 중 open 은 무시
print("상태 기계 OK")

## 4. 웨이크 워드 리스너 (항상 켜져 있다)

`01_wakeword.ipynb` 7장의 `stream_detect()` 를 무한 실행 스레드로 바꾼 것이다.
콜백에서는 큐에 넣기만 하고 추론은 루프에서 한다 — 콜백 안에서 추론하면 오디오가 드롭된다.

이 스레드는 **DPU 를 전혀 만지지 않는다.** 그래서 xmodel 교체 중에도, 제스처 추론이
DPU 를 붙잡고 있어도 계속 듣는다.

In [ ]:
import sounddevice as sd
from openwakeword.model import Model


class WakeListener(threading.Thread):
    """마이크를 계속 듣다가 웨이크 워드가 잡히면 event_q 에 "wake" 를 넣는다."""

    def __init__(self, event_q):
        super().__init__(daemon=True)
        self.event_q = event_q
        self.q = queue.Queue()
        self.score = 0.0
        self.n_wake = 0
        self.drops = 0
        self.error = None
        self._running = False

    def _callback(self, indata, frames, time_info, status):
        if status:
            self.drops += 1
        self.q.put(indata[:, 0].copy())

    def run(self):
        self._running = True
        try:
            oww = Model(wakeword_models=[WAKEWORD_MODEL], inference_framework="onnx")
            key = list(oww.models.keys())[0]
            last_fire = -1e9
            with sd.InputStream(samplerate=SAMPLE_RATE, channels=CHANNELS, dtype=DTYPE,
                                blocksize=CHUNK, device=MIC_INDEX, callback=self._callback):
                print(f"[MIC ] 듣는 중 — \"{key}\"")
                while self._running:
                    try:
                        frame = self.q.get(timeout=0.5)
                    except queue.Empty:
                        continue
                    self.score = float(oww.predict(frame)[key])
                    now = time.monotonic()
                    if self.score > WAKE_THRESHOLD and (now - last_fire) > WAKE_REFRACTORY:
                        last_fire = now
                        self.n_wake += 1
                        print(f"[WAKE] score={self.score:.3f}")
                        self.event_q.put("wake")
        except Exception as e:
            self.error = f"{type(e).__name__}: {e}"
            print(f"[MIC ] 죽었다 — {self.error}")

    def stop(self):
        self._running = False
        self.join(timeout=3.0)

## 5. DPU 워커 (하나뿐인 DPU 를 독점한다)

모드 전환 = `rt.load_model()`. 비트스트림은 `Runner()` 생성 때 한 번만 올라간다.

교체할 때 **시퀀스 버퍼를 반드시 비운다.** 안 비우면 교체 직전 1초 창이 살아남아
제스처 모드로 돌아오는 순간 `open` 이 다시 발화하고 곧바로 TASK 로 튕겨 나간다.

`SequenceBuffer` 는 프레임 수가 아니라 시간 창(`window_s`)으로 30점을 리샘플링하므로,
이 루프가 카메라보다 느려도 GRU 가 보는 속도 특징은 유지된다. 손댈 곳이 없다.

In [ ]:
class DpuWorker(threading.Thread):
    """모드에 따라 xmodel 을 갈아 끼우며 해당 카메라에 추론을 돌린다."""

    def __init__(self, event_q, cams):
        super().__init__(daemon=True)
        self.event_q = event_q
        self.cams = cams            # [cam0(제스처), cam1(작업)]
        self.mode = "IDLE"
        self.loaded = os.path.basename(GESTURE_XMODEL)
        self.det = None             # {"cam": i, "boxes"/"scores"/"cls_ids"}
        self.dpu_ms = 0.0
        self.swap_ms = 0.0
        self.gesture = ""
        self.error = None
        self.ready = False
        self._running = False

    def _drain(self, mode):
        while True:
            try:
                ev = self.event_q.get_nowait()
            except queue.Empty:
                return mode
            new = next_mode(mode, ev)
            if new != mode:
                print(f"[MODE] {mode} -> {new}   (event={ev})")
            mode = new

    def run(self):
        self._running = True

        # PYNQ 는 인터럽트를 asyncio 로 다뤄서 오버레이를 만질 때 이벤트 루프를 요구한다.
        # 워커 스레드에는 루프가 없고, 3.10+ 부터는 자동 생성도 안 해준다:
        #   RuntimeError: There is no current event loop in thread 'Thread-N'
        # 교체(load_model)도 이 스레드에서 도니 생성만 메인 스레드로 옮겨서는 못 고친다.
        import asyncio
        asyncio.set_event_loop(asyncio.new_event_loop())

        try:
            rt = Runner(BIT_PATH, GESTURE_XMODEL)
            gru = GestureClassifier(GRU_PATH)
            print(f"[DPU ] input={rt.in_dims} layout={rt.layout} int8={rt.in_is_int8}")
        except Exception as e:
            self.error = f"{type(e).__name__}: {e}"
            print(f"[DPU ] 초기화 실패 — {self.error}")
            return

        seqbuf = SequenceBuffer(window_s=WINDOW_S)
        mode = "IDLE"
        loaded_mode = "GESTURE"      # Runner 가 이미 제스처 xmodel 을 올려 뒀다
        n, last_fire = 0, -1e9
        self.ready = True

        while self._running:
            mode = self._drain(mode)
            self.mode = mode

            if mode == "IDLE":       # 웨이크 워드 전에는 DPU 를 아예 쓰지 않는다
                time.sleep(0.05)
                continue

            if mode != loaded_mode:
                want = GESTURE_XMODEL if mode == "GESTURE" else TASK_XMODEL
                t0 = time.perf_counter()
                rt.load_model(want)
                self.swap_ms = (time.perf_counter() - t0) * 1000
                seqbuf.buf.clear()               # 교체 전 창을 버린다
                self.det = None
                self.gesture = ""
                loaded_mode = mode
                self.loaded = os.path.basename(want)
                print(f"[SWAP] {self.loaded} 로 교체  {self.swap_ms:.0f} ms")

            idx = 0 if mode == "GESTURE" else 1
            frame = self.cams[idx].snapshot()
            if frame is None:
                time.sleep(0.01)
                continue

            inp, ratio, pad, orig_wh = rt.preprocess(frame)
            t0 = time.perf_counter()
            outputs = rt.infer(inp)
            boxes, scores, cls_ids = decode(outputs, ratio, pad, orig_wh, CONF)
            self.dpu_ms = (time.perf_counter() - t0) * 1000
            self.det = {"cam": idx, "boxes": boxes, "scores": scores, "cls_ids": cls_ids}

            if mode != "GESTURE":
                continue             # 작업 모델은 그리기만 한다 (대역이라 후처리가 같다)

            # ---- 2단계: 특징 시퀀스 -> GRU (gesture_live.main 과 같은 규칙) ----
            now = time.time()
            if len(boxes):
                x1, y1, x2, y2 = boxes[0]
                ow, oh = orig_wh
                det = ((x1 + x2) / 2 / ow, (y1 + y2) / 2 / oh,
                       (x2 - x1) / ow, (y2 - y1) / oh,
                       float(scores[0]), int(cls_ids[0]))
            else:
                det = None
            seqbuf.push(now, det)

            n += 1
            if n % GRU_EVERY or not seqbuf.ready(now):
                continue

            seq = build_sequence(seqbuf.sample(now))
            if seq is None:
                continue

            active, disp, shape = is_active(seq)
            if not active:                       # 정지 게이트: 손이 멈춰 있으면 GRU 를 안 부른다
                self.gesture = ""
                continue

            prob = gru.predict(seq)
            k = int(np.argmax(prob))
            if prob[k] < GESTURE_THRES or (now - last_fire) < GESTURE_REFRACTORY:
                continue

            last_fire = now
            self.gesture = f"{GESTURE_CLASSES[k]} {prob[k]:.2f}"
            print(f"[GEST] {GESTURE_CLASSES[k]:12s} p={prob[k]:.3f}")
            if GESTURE_CLASSES[k] == TRIGGER:
                self.event_q.put("open")         # 직접 대입하지 않고 같은 큐로 보낸다

    def stop(self):
        self._running = False
        self.join(timeout=5.0)

## 6. 실행

셀을 실행하면 화면 두 개가 뜬다. **Stop 버튼으로 종료한다.** 셀을 중단(interrupt)하지 말 것.

초록 테두리가 지금 DPU 가 보고 있는 카메라다.

1. 시작 직후 — `IDLE`. 두 화면 다 테두리 없음. DPU 유휴
2. 웨이크 워드 → `GESTURE`. cam0 에 손 박스가 뜬다
3. `open` 제스처 → `[SWAP]` 이 찍히고 박스가 cam1 로 넘어간다
4. 다시 웨이크 워드 → cam0 으로 복귀

`Wake (수동)` 버튼은 마이크가 말을 안 들을 때 같은 이벤트를 직접 넣는다. 시연용 보험이다.

In [ ]:
streams, view, worker, listener = [], None, None, None
event_q = queue.Queue()

MODE_COLOR = {"IDLE": (120, 120, 120), "GESTURE": (0, 220, 120), "TASK": (0, 180, 255)}


def overlay(frame, text, active, color):
    """좌상단 라벨 + 활성 카메라 테두리."""
    out = frame.copy()
    cv2.rectangle(out, (0, 0), (330, 34), (0, 0, 0), -1)
    out = cv2.addWeighted(out, 0.65, frame, 0.35, 0)
    cv2.putText(out, text, (8, 24), cv2.FONT_HERSHEY_SIMPLEX,
                0.6, (0, 255, 120), 1, cv2.LINE_AA)
    if active:
        h, w = out.shape[:2]
        cv2.rectangle(out, (0, 0), (w - 1, h - 1), color, 4)
    return out


def start():
    global streams, view, worker, listener

    stop_all()
    while not event_q.empty():
        event_q.get_nowait()

    streams = [CamStream(CAM_A_ID, "CAM_A 사용자"), CamStream(CAM_B_ID, "CAM_B 대상")]
    for s in streams:
        if not s.open():
            print(f"[FAIL] {s.label} (/dev/video{s.dev_id}) 열기 실패")
            print("       dmesg | grep -i 'Not enough bandwidth' 로 확인할 것")
            stop_all()
            return
        s.start()
    time.sleep(0.5)

    listener = WakeListener(event_q)
    listener.start()
    worker = DpuWorker(event_q, streams)
    worker.start()

    imgs = [widgets.Image(format="jpeg") for _ in streams]
    caps = [widgets.HTML(f"<b>{s.label}</b>") for s in streams]
    cols = [widgets.VBox([c, i]) for c, i in zip(caps, imgs)]

    btn_stop = widgets.Button(description="Stop", button_style="danger", icon="stop")
    btn_wake = widgets.Button(description="Wake (수동)", button_style="info")
    status = widgets.HTML("시작 중… (DPU 비트스트림 로드에 몇 초 걸린다)")

    view = {"running": True}
    btn_wake.on_click(lambda _: event_q.put("wake"))

    def on_stop(_):
        view["running"] = False
        btn_stop.disabled = True
        btn_stop.description = "Stopped"

    btn_stop.on_click(on_stop)
    display(widgets.VBox([widgets.HBox(cols), widgets.HBox([btn_stop, btn_wake, status])]))

    def loop(state):
        period = 1.0 / DISPLAY_FPS
        while state["running"]:
            t0 = time.monotonic()
            mode = worker.mode
            color = MODE_COLOR[mode]
            det = worker.det
            for i, (w, s) in enumerate(zip(imgs, streams)):
                f = s.snapshot()
                if f is None:
                    continue
                active = (i == 0 and mode == "GESTURE") or (i == 1 and mode == "TASK")
                if active and det is not None and det["cam"] == i:
                    draw(f, det["boxes"], det["scores"], det["cls_ids"])   # 축소 전에 그린다
                if DISPLAY_SCALE != 1.0:
                    f = cv2.resize(f, None, fx=DISPLAY_SCALE, fy=DISPLAY_SCALE,
                                   interpolation=cv2.INTER_AREA)
                f = overlay(f, f"{s.label}  {s.fps:4.1f} fps", active, color)
                ok, buf = cv2.imencode(".jpg", f, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
                if ok:
                    w.value = buf.tobytes()

            status.value = (
                f"&nbsp;&nbsp;<b>{mode}</b> · {worker.loaded} · "
                f"DPU {worker.dpu_ms:.0f}ms · swap {worker.swap_ms:.0f}ms · "
                f"제스처 {worker.gesture or '—'} · "
                f"wake {listener.n_wake}회 (score {listener.score:.2f}, drop {listener.drops})"
                + (f" · <span style='color:crimson'>{worker.error or listener.error}</span>"
                   if (worker.error or listener.error) else "")
            )
            time.sleep(max(0.0, period - (time.monotonic() - t0)))

        stop_all()
        status.value = "&nbsp;&nbsp;정지됨. 카메라·마이크·DPU 를 해제했다."

    threading.Thread(target=loop, args=(view,), daemon=True).start()


def stop_all():
    global streams, view, worker, listener
    if view is not None:
        view["running"] = False
        view = None
    for obj in [listener, worker, *streams]:
        try:
            obj.stop()
        except Exception:
            pass
    listener, worker, streams = None, None, []


In [ ]:
start()

## 7. 정리

셀을 중단했거나 커널을 재시작하기 전에 실행한다.
카메라나 마이크가 열린 채 남으면 다음 실행에서 열기가 실패한다.

In [ ]:
stop_all()
print("카메라·마이크·DPU 해제 완료")

## 8. 문제 해결

### `[MODE]` 는 찍히는데 `[SWAP]` 이 안 찍힌다

`IDLE -> GESTURE` 는 교체가 아니다. `Runner()` 가 이미 제스처 xmodel 을 올려 둔 상태라
일부러 건너뛴다. `GESTURE -> TASK` 에서만 실제 교체가 일어난다.

### 오디오 드롭(`drop`)이 0 이 아니다

DPU 추론과 카메라 디코딩이 A53 4코어를 나눠 쓰면서 오디오 스레드가 굶은 것이다.
순서대로 내린다.

```python
DISPLAY_FPS = 6        # 먼저 이것
DISPLAY_SCALE = 0.4
GRU_EVERY = 8          # 그래도 안 되면
```

01번에서 잰 `cpu_load_pct` 와 비교한다. 웨이크 워드 예산은 청크당 80 ms 다.

### 두 번째 카메라가 안 열린다

```bash
dmesg | grep -iE "Not enough bandwidth|Capping" | tail
sudo /usr/local/sbin/uvc-swap.sh on 1024
```

### 제스처가 안 잡힌다

`gesture_live.py --gate-debug` 로 단독 확인하는 편이 빠르다. 정지 게이트(`is_active`)가
막고 있으면 손을 크게 움직이거나(`disp > 0.04`) 손모양을 바꾼다(`shape > 0.5`).

박스 자체가 안 뜨면 게이트가 아니라 1단계 문제다. 카메라 거리와 조명을 본다.

### `open` 이 교체 직후 다시 발화한다

`seqbuf.buf.clear()` 가 빠진 경우다. 5장 워커에 이미 들어 있으니 지우지 말 것.

### 교체가 느리다

`[SWAP]` 의 ms 를 본다. 비트스트림까지 다시 올리면 초 단위가 되지만 여기서는
`overlay.load_model()` 만 부른다. 초 단위가 나온다면 `Runner` 를 새로 만들고 있는 것이다.

---

### 다음 단계

- `TASK_XMODEL` 을 진짜 두 번째 모델로 바꾸고, 그 모델의 후처리를 워커의 `mode != "GESTURE"`
  분기에 넣는다 (지금은 대역이라 제스처 `decode()` 를 그대로 쓴다).
- 커스텀 `hey_kria.onnx` 가 나오면 `WAKEWORD_MODEL` 을 경로로 바꾼다.
- 상태가 셋을 넘어가면 그때 `next_mode()` 를 테이블로 바꾼다. 둘까지는 지금이 더 짧다.